In [6]:
import numpy as np
import time
import math
import multiprocessing as mp
import dask.array as da
from dask.distributed import Client, LocalCluster
from numba import cuda
import matplotlib.pyplot as plt
import unittest

In [7]:
def mandelbrot_naive(width, height, max_iter=100):
    """
    Computes the Mandelbrot set using a naive, sequential nested loop.
    
    This function iterates point-by-point over the complex plane, calculating
    z = z^2 + c. It is intended to serve as a baseline for performance 
    comparisons against parallelized and hardware-accelerated versions.

    Parameters:
    -----------
    width : int
        The number of pixels in the x-dimension.
    height : int
        The number of pixels in the y-dimension.
    max_iter : int, optional
        Maximum number of iterations before assuming the point is bounded (default 100).

    Returns:
    --------
    numpy.ndarray
        A 2D array of floats representing the normalized iteration count for each pixel.
    """
    result = np.zeros((height, width), dtype=float)
    x_coords = np.linspace(-2.0, 1.0, width)
    y_coords = np.linspace(-1.5, 1.5, height)
    
    for j in range(height):
        for i in range(width):
            c = complex(x_coords[i], y_coords[j])
            z = 0j
            for n in range(max_iter):
                if abs(z) > 2.0:
                    result[j, i] = n / max_iter
                    break
                z = z*z + c
            else:
                result[j, i] = 1.0
    return result

def mandelbrot_numpy(c_grid, max_iter=100):
    """
    Computes the Mandelbrot set using vectorized NumPy operations.
    """
    z = np.zeros_like(c_grid, dtype=complex)
    iterations = np.zeros(c_grid.shape, dtype=float)
    
    for n in range(max_iter):
        mask = np.abs(z) <= 2
        if not np.any(mask):
            break
        z[mask] = z[mask]**2 + c_grid[mask]
        iterations[mask] = n
        
    return iterations / max_iter

In [8]:
# --- Multiprocessing ---
def mandel_pixel_calc(c, max_iter):
    z = 0j
    for n in range(max_iter):
        if abs(z) > 2.0:
            return n / max_iter
        z = z*z + c
    return 1.0

def mandelbrot_multiprocessing(width, height, max_iter=100, processes=4, chunksize=100):
    x = np.linspace(-2.0, 1.0, width)
    y = np.linspace(-1.5, 1.5, height)
    X, Y = np.meshgrid(x, y)
    c_flat = (X + 1j*Y).flatten()
    
    with mp.Pool(processes=processes) as pool:
        result = pool.starmap(mandel_pixel_calc, [(c, max_iter) for c in c_flat], chunksize=chunksize)
        
    return np.array(result).reshape((height, width))

# --- Dask ---
def run_dask_mandelbrot(width, height, chunk_size, max_iter=100):
    x = da.linspace(-2.0, 1.0, width, chunks=chunk_size)
    y = da.linspace(-1.5, 1.5, height, chunks=chunk_size)
    X, Y = da.meshgrid(x, y)
    c_grid = X + 1j*Y
    
    mandel_dask = c_grid.map_blocks(mandelbrot_numpy, max_iter=max_iter, dtype=float)
    return mandel_dask.compute()

In [9]:
@cuda.jit
def mandelbrot_kernel(d_iterations, min_x, max_x, min_y, max_y, width, height, max_iter):
    """
    CUDA kernel to compute the Mandelbrot set on the GPU.
    """
    # 2D Grid/Block mapping
    x_idx, y_idx = cuda.grid(2)

    # Out-of-bounds check
    if x_idx < width and y_idx < height:
        pixel_size_x = (max_x - min_x) / width
        pixel_size_y = (max_y - min_y) / height
        
        c_real = min_x + x_idx * pixel_size_x
        c_imag = min_y + y_idx * pixel_size_y
        
        z_real = 0.0
        z_imag = 0.0
        
        for i in range(max_iter):
            if z_real * z_real + z_imag * z_imag >= 4.0:
                break
            
            temp_real = z_real * z_real - z_imag * z_imag + c_real
            z_imag = 2.0 * z_real * z_imag + c_imag
            z_real = temp_real
            
        d_iterations[y_idx, x_idx] = i / max_iter

def run_cuda_mandelbrot(width, height, threads_per_block=(16, 16), max_iter=100):
    # Data transfer: Allocate device memory
    d_iterations = cuda.device_array((height, width), dtype=np.float64)
    
    blocks_per_grid_x = math.ceil(width / threads_per_block[0])
    blocks_per_grid_y = math.ceil(height / threads_per_block[1])
    blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)
    
    cuda.synchronize() # Synchronization
    start_time = time.time()
    
    mandelbrot_kernel[blocks_per_grid, threads_per_block](
        d_iterations, -2.0, 1.0, -1.5, 1.5, width, height, max_iter
    )
    
    cuda.synchronize()
    execution_time = time.time() - start_time
    
    # Data transfer: Copy back to CPU
    h_iterations = d_iterations.copy_to_host()
    
    return h_iterations, execution_time